# FLAG 2027 - Notebook 27: within-gender discrimination (the gender-English cell)

System C on dev (1 Nov): 20.44 / 20.34 / **35.23** / 30.79. The same-gender English cell is our worst and the one where the honest teams above us are 5 points better. Four same-gender-focused *training* changes already failed in CV (gender-hard negatives NB03, same-gender negatives x3 NB09, gender-balanced batches NB15, gender-consistency term NB15) and LEACE levelled down. This notebook tests the two things that were NOT tried, both legal (per-trial, training data only, no test statistics):

| Combo | What changes | Why it might work where the others failed |
|---|---|---|
| `G0_C_ref` | system C recipe (R4 + Whisper + NAP-8), unchanged | reference, re-run in this harness (NB24 D4 = 25.15 / gender 31.28) |
| `G1_within_CE` | identity cross-entropy where only classes of the sample's OWN gender compete (other-gender logits masked) | the head is forced to separate identities within gender; NB09's x3 weighting was a soft version of this |
| `G2_all+within_CE` | both losses summed | keeps the cross-gender signal for the standard track |
| `G3_gender_experts` | two heads, each trained on one gender's identities only; each trial is scored by the head of the FACE's gender (CLIP gender probe, identity-disjoint accuracy printed) | an expert never sees the gender shortcut at all; risk: half the identities per head, and the two heads' score scales must match without any normalisation (diagnostic `gender_pg` shows the ceiling) |

Gender of the challenge identities: official meta file. Gender of the MAV identities: CLIP logistic probe trained on the 70 challenge identities (accuracy printed; expected > 0.99). 5-fold x 3 seeds, no VoxCeleb (stage 1). About 3 h, resumable. **Decision rule:** adopt only if `gender` improves by >= 0.5 over G0 AND `std` is not worse by more than 0.2; then stage 2 = NB20 v5 with VoxCeleb. Paste the **SUMMARY**.

In [ ]:
# Cell 1 - Drive, GPU (hard check), packages
from google.colab import drive
drive.mount('/content/drive')
import torch, shutil
assert torch.cuda.is_available(), 'NO GPU in this runtime. Runtime > Change runtime type > T4 GPU > Save, then run Cell 1 again.'
dev = 'cuda'
print('[v1 1-Nov] Device:', torch.cuda.get_device_name(0), '| disk free %.0f GB' % (shutil.disk_usage('/content')[2] / 1e9))

In [ ]:
# Cell 2 - features, labels, CV folds (identical to NB24), gender of every training identity, CV harness
import os, re, json, time, zipfile, copy
import numpy as np, pandas as pd
import torch.nn as nn, torch.nn.functional as Fn
from sklearn.metrics import roc_curve
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold, cross_val_score
BASE = '/content/drive/MyDrive'
PROJECT = os.path.join(BASE, 'FLAG2027'); FEATDIR = os.path.join(PROJECT, 'features'); RESDIR = os.path.join(PROJECT, 'results')
ZIP_DIR = json.load(open(os.path.join(PROJECT, 'paths.json')))['ZIP_DIR']
SUMMARY = []
def note(msg):
    print(msg); SUMMARY.append(msg)
LISTS = '/content/flag_lists'
if not os.path.exists(LISTS + '/done'):
    with zipfile.ZipFile(os.path.join(ZIP_DIR, 'train_set.zip')) as zf:
        zf.extractall(LISTS, members=[m for m in zf.namelist() if m.endswith(('.txt', '.csv'))])
    open(LISTS + '/done', 'w').close()
TXT = [os.path.join(r, f) for r, _, fs in os.walk(LISTS) for f in fs if f.endswith(('.txt', '.csv'))]
def find_one(*parts):
    hits = [p for p in TXT if all(x in p for x in parts)]
    assert len(hits) == 1, 'looking for %s -> found %s' % (parts, hits)
    return hits[0]
def read_feat(path, dim):
    df = pd.read_csv(path, header=None)
    return df.iloc[:, :dim].values.astype(np.float32), (df.iloc[:, dim].values if df.shape[1] > dim else None)
Xv, y = read_feat(find_one('train_English_voices.csv'), 192); y = y.astype(int)
train_lines = [l.split() for l in open(find_one('train_English.txt')) if l.strip()]
cls2spk = {int(l[5]): l[4] for l in train_lines}
meta = pd.read_csv(os.path.join(ZIP_DIR, 'meta_file_train_set.csv'), header=None, names=['spk', 'g'])
spk2g = dict(zip(meta.spk.astype(str).str.strip(), meta.g.astype(str).str.strip().str.lower()))
ids = np.unique(y); id_gender = {c: spk2g[cls2spk[c]] for c in ids}
GENDERS = sorted(set(id_gender.values())); assert len(GENDERS) == 2, GENDERS; g2i = {g: i for i, g in enumerate(GENDERS)}
MAV = dict(np.load(os.path.join(FEATDIR, 'mavceleb_v1v3.npz'))); EXTRA = dict(np.load(os.path.join(FEATDIR, 'mavceleb_extra.npz')))
ML = np.load(os.path.join(FEATDIR, 'mav_lang.npz')); EXT_SPK, EXT_LANG = ML['spk'], ML['lang']
assert list(EXT_SPK) == list(MAV['spk']) + list(EXTRA['spk']), 'mav_lang.npz does not match the feature files - tell Claude'
FEAT = {'ecapa': {'train': Xv, 'mav': MAV['ecapa'], 'extra': EXTRA['ecapa']}}
for name in ['clip', 'wavlm_sv', 'r293', 'redim', 'audeer', 'whisper']:
    d = np.load(os.path.join(FEATDIR, name + '.npz')); FEAT[name] = {s: d[s] for s in d.files if not s.endswith('_ag')}
FEAT['clip']['mav'], FEAT['clip']['extra'] = MAV['clip'], EXTRA['clip']; FEAT['wavlm_sv']['mav'], FEAT['wavlm_sv']['extra'] = MAV['wavlm_sv'], EXTRA['wavlm_sv']
FEAT['r293']['extra'] = EXTRA['r293']
for name in FEAT:
    for s in ['train', 'mav', 'extra']:
        assert s in FEAT[name] and len(FEAT[name][s]) == {'train': len(y), 'mav': len(MAV['spk']), 'extra': len(EXTRA['spk'])}[s], '%s/%s missing or wrong length - tell Claude' % (name, s)
N_CH = len(y); ext_spk = list(EXT_SPK)
ext_lab = {s: 1000 + i for i, s in enumerate(sorted(set(ext_spk)))}
Y_ALL = np.r_[y, np.array([ext_lab[s] for s in ext_spk])]; D_ALL = np.r_[np.zeros(N_CH, int), np.ones(len(ext_spk), int)]; EXT = N_CH + np.arange(len(ext_spk))
def l2(X):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + 1e-9)
def build(names, split):
    return np.concatenate([l2(FEAT[n][split]) for n in names], 1).astype(np.float32)
def build_train(names):
    return np.concatenate([build(names, 'train'), build(names, 'mav'), build(names, 'extra')])
# ---- gender of every training identity: challenge = official meta; MAV = CLIP logistic probe (majority over the identity's rows) ----
F_CLIP = build_train(['clip']); yg_ch = np.array([g2i[id_gender[c]] for c in y])
acc = cross_val_score(LogisticRegression(C=1.0, max_iter=3000), F_CLIP[:N_CH], yg_ch, groups=y, cv=GroupKFold(5)).mean()
pg_rows = LogisticRegression(C=1.0, max_iter=3000).fit(F_CLIP[:N_CH], yg_ch).predict(F_CLIP[N_CH:])
CLASS_G = {c: g2i[id_gender[c]] for c in ids}; agree = []
for s, c in ext_lab.items():
    p = pg_rows[EXT_SPK == s].mean(); CLASS_G[c] = int(p > 0.5); agree.append(max(p, 1 - p))
n0 = sum(CLASS_G[c] == 0 for c in ext_lab.values()); n1 = len(ext_lab) - n0
note('gender probe (CLIP, logistic): identity-disjoint CV accuracy on the 70 challenge ids %.3f | MAV identities: %s %d, %s %d | mean within-identity agreement %.3f' % (acc, GENDERS[0], n0, GENDERS[1], n1, np.mean(agree)))
assert acc > 0.97, 'gender probe too weak - tell Claude'
# ---- NAP (as NB24 / NB20 v4) ----
NAP_K = 8
def nap_basis(V_ext, k):
    diffs = []
    for s in np.unique(EXT_SPK):
        m = EXT_SPK == s; langs = np.unique(EXT_LANG[m])
        if len(langs) < 2:
            continue
        mu = [V_ext[m & (EXT_LANG == l)].mean(0) for l in langs]
        for a in range(len(mu)):
            for b in range(a + 1, len(mu)):
                diffs.append(mu[a] - mu[b])
    Dm = np.stack(diffs); U = np.linalg.svd(Dm - Dm.mean(0), full_matrices=False)[2][:k]
    return U.astype(np.float32), len(diffs)
def nap_apply(X, U):
    return (X - (X @ U.T) @ U).astype(np.float32)
# ---- CV harness (identical folds / pairs / model / losses to NB24 Cell 4) ----
def eer(labels, scores):
    fpr, tpr, _ = roc_curve(labels, scores); fnr = 1 - tpr
    i = np.nanargmin(np.abs(fnr - fpr)); return (fpr[i] + fnr[i]) / 2 * 100
def make_pairs(idx, n=2000, same_gender_neg=False, seed=0):
    rng = np.random.default_rng(seed); spk_ = np.unique(y[idx]); rows_ = {s: idx[y[idx] == s] for s in spk_}
    V, F, L = [], [], []
    for _ in range(n):
        s = rng.choice(spk_); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[s])); L.append(1)
        s = rng.choice(spk_)
        cand = [t for t in spk_ if t != s and (not same_gender_neg or id_gender[t] == id_gender[s])] or [t for t in spk_ if t != s]
        t = rng.choice(cand); V.append(rng.choice(rows_[s])); F.append(rng.choice(rows_[t])); L.append(0)
    return np.array(V), np.array(F), np.array(L)
K = 5; rng = np.random.default_rng(42); folds = [[] for _ in range(K)]
for gg in sorted(set(id_gender.values())):
    members = [c for c in ids if id_gender[c] == gg]; rng.shuffle(members)
    for i, c in enumerate(members):
        folds[i % K].append(c)
FOLD_DATA, FACE_G = [], []
for k in range(K):
    val_ids = set(folds[k]); va_idx = np.where([c in val_ids for c in y])[0]; tr_idx = np.where([c not in val_ids for c in y])[0]
    FOLD_DATA.append((tr_idx, [make_pairs(va_idx, seed=k), make_pairs(va_idx, same_gender_neg=True, seed=100 + k)]))
    FACE_G.append(LogisticRegression(C=1.0, max_iter=3000).fit(F_CLIP[tr_idx], yg_ch[tr_idx]).predict(F_CLIP[:N_CH]))   # face gender for routing, fitted on the fold's TRAIN identities only
fg_acc = np.mean([(FACE_G[k][np.where([c in set(folds[k]) for c in y])[0]] == yg_ch[np.where([c in set(folds[k]) for c in y])[0]]).mean() for k in range(K)])
note('face-gender routing accuracy on held-out identities (per fold probe): %.4f' % fg_acc)
_PROTO = {}
def max_sep_prototypes(n):
    if n in _PROTO:
        return _PROTO[n]
    P = np.array([[1.0, -1.0]])
    for k in range(2, n):
        Q = np.zeros((k, k + 1)); Q[0, 0] = 1.0; Q[0, 1:] = -1.0 / k; Q[1:, 1:] = np.sqrt(1 - 1.0 / k ** 2) * P; P = Q
    _PROTO[n] = torch.tensor(P.T, dtype=torch.float32); return _PROTO[n]
class Branch(nn.Module):
    def __init__(self, d_in, d=128):
        super().__init__(); self.net = nn.Sequential(nn.Linear(d_in, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Dropout(0.5))
    def forward(self, x):
        return Fn.normalize(self.net(x), dim=1)
class FOP(nn.Module):
    def __init__(self, n_class, d_face, d_voice, d=128):
        super().__init__()
        self.face, self.voice = Branch(d_face, d), Branch(d_voice, d)
        self.att = nn.Sequential(nn.Linear(2 * d, d), nn.BatchNorm1d(d), nn.ReLU(), nn.Linear(d, d))
        self.cls = nn.Linear(d, n_class - 1); self.register_buffer('P', max_sep_prototypes(n_class))
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight); nn.init.constant_(m.bias, 0.01)
    def embed(self, xf, xv):
        f, v = self.face(xf), self.voice(xv); return torch.tanh(f), torch.tanh(v), f, v
    def forward(self, xf, xv):
        ft, vt, f, v = self.embed(xf, xv); a = torch.sigmoid(self.att(torch.cat([f, v], 1))); fused = ft * a + (1 - a) * vt
        return fused, self.cls(fused) @ self.P.t()
def opl_loss(feat, lab):
    feat = Fn.normalize(feat, dim=1); dot = feat @ feat.t(); same = (lab[:, None] == lab[None, :]).float()
    pos = same - torch.eye(len(lab), device=feat.device); neg = 1 - same
    return (1 - (pos * dot).sum() / (pos.sum() + 1e-6)) + 0.7 * (neg * dot.abs()).sum() / (neg.sum() + 1e-6)
def masked_ce(logits, lab, cg):
    """cross-entropy where only the classes of the sample's own gender compete"""
    g = cg[lab]; mask = cg[None, :] != g[:, None]
    return Fn.cross_entropy(logits.masked_fill(mask, -1e4), lab)
@torch.no_grad()
def same_score(model, xf, xv):
    model.eval(); mf, sf_, mv, sv = model.norm; ft, vt, _, _ = model.embed((xf - mf) / sf_, (xv - mv) / sv)
    return -(ft - vt).norm(dim=1).cpu().numpy()
def dnorm(F_raw, V_raw, tr_idx):
    F, V = F_raw.copy(), V_raw.copy(); ch = None
    for d in np.unique(D_ALL[tr_idx]):
        trd = tr_idx[D_ALL[tr_idx] == d]; alld = np.where(D_ALL == d)[0]
        st = (F_raw[trd].mean(0), F_raw[trd].std(0) + 1e-6, V_raw[trd].mean(0), V_raw[trd].std(0) + 1e-6)
        F[alld] = (F_raw[alld] - st[0]) / st[1]; V[alld] = (V_raw[alld] - st[2]) / st[3]
        if d == 0:
            ch = st
    return F, V, ch
SEEDS = [1, 2, 3]; P_ID, K_CLIP, ALPHA, LR, MAX_EP, PATIENCE, STEPS = 16, 8, 2.0, 1e-4, 100, 15, 60
def train_head(F_raw, V_raw, all_tr, vsets=None, seed=1, max_epochs=MAX_EP, mode='plain'):
    """mode: 'plain' = CE over all classes (system C recipe); 'within' = CE among same-gender classes only; 'both' = sum of the two"""
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    F, V, ch = dnorm(F_raw, V_raw, all_tr); XF = torch.tensor(F, device=dev); XV = torch.tensor(V, device=dev)
    cl = np.unique(Y_ALL[all_tr]); rm = {c: i for i, c in enumerate(cl)}; cg = torch.tensor([CLASS_G[c] for c in cl], device=dev)
    ytr_np = np.array([rm[c] for c in Y_ALL[all_tr]]); ytr = torch.tensor(ytr_np, device=dev); pos_by_cls = [np.where(ytr_np == c)[0] for c in range(len(cl))]
    tr = torch.tensor(all_tr, device=dev); model = FOP(len(cl), XF.shape[1], XV.shape[1]).to(dev)
    model.norm = (XF[tr].mean(0), XF[tr].std(0) + 1e-6, XV[tr].mean(0), XV[tr].std(0) + 1e-6); mf, sf_, mv, sv = model.norm
    opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=0.01); best, best_ep, best_state, wait = 1e9, max_epochs, None, 0
    for ep in range(1, max_epochs + 1):
        model.train()
        for _ in range(STEPS):
            chosen = rng.choice(len(cl), P_ID, replace=False)
            j = torch.tensor(np.concatenate([rng.choice(pos_by_cls[c], K_CLIP, replace=len(pos_by_cls[c]) < K_CLIP) for c in chosen]), device=dev)
            fused, logits = model((XF[tr[j]] - mf) / sf_, (XV[tr[j]] - mv) / sv)
            if mode == 'plain':
                ce = Fn.cross_entropy(logits, ytr[j])
            elif mode == 'within':
                ce = masked_ce(logits, ytr[j], cg)
            else:
                ce = Fn.cross_entropy(logits, ytr[j]) + masked_ce(logits, ytr[j], cg)
            loss = ce + ALPHA * opl_loss(fused, ytr[j]); opt.zero_grad(); loss.backward(); opt.step()
        if vsets is None:
            continue
        score = np.mean([eer(L, same_score(model, XF[Fi], XV[Vi])) for Vi, Fi, L in vsets if len(L)])
        if score < best - 0.05:
            best, best_ep, best_state, wait = score, ep, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_ep, XF, XV, ch
R4 = ['ecapa', 'wavlm_sv', 'r293', 'redim', 'audeer']; VOICE = R4 + ['whisper']
F_RAW = build_train(['clip']); V_full = build_train(VOICE); U_NAP, n_d = nap_basis(V_full[N_CH:], NAP_K); V_RAW = nap_apply(V_full, U_NAP)
Y_G = np.array([CLASS_G[c] for c in Y_ALL])          # gender of every training row
note('harness ready: %d challenge rows, %d MAV rows, voice %d-d (NAP-%d from %d language-difference vectors) | identities per gender in training: %s %d, %s %d' % (N_CH, len(EXT), V_RAW.shape[1], NAP_K, n_d, GENDERS[0], sum(v == 0 for v in CLASS_G.values()), GENDERS[1], sum(v == 1 for v in CLASS_G.values())))

In [ ]:
# Cell 3 - CV: reference, within-gender CE (two variants), gender experts (5 folds x 3 seeds; resumable) -> table + verdict
COMBOS = {'G0_C_ref': 'plain', 'G1_within_CE': 'within', 'G2_all+within_CE': 'both', 'G3_gender_experts': 'experts'}
CSV = os.path.join(RESDIR, 'nb27_cv.csv')
done = pd.read_csv(CSV) if os.path.exists(CSV) else pd.DataFrame(columns=['combo', 'seed', 'fold']); t0 = time.time()
def score_sets(parts, vsets, fg):
    """parts: {gender: (model, XF, XV)} or {None: (model, XF, XV)}; every trial is scored by the head of its FACE's predicted gender"""
    out = []
    for Vi, Fi, L in vsets:
        s = np.zeros(len(Fi), np.float32)
        for g, (m_, XF, XV) in parts.items():
            sel = np.ones(len(Fi), bool) if g is None else fg[Fi] == g
            if sel.any():
                s[sel] = same_score(m_, XF[Fi[sel]], XV[Vi[sel]])
        out.append(s)
    return out
for name, mode in COMBOS.items():
    for seed in SEEDS:
        for k in range(K):
            if ((done.combo == name) & (done.seed == seed) & (done.fold == k)).any():
                continue
            tr_idx, vsets = FOLD_DATA[k]; fg = FACE_G[k]; cal = float('nan')
            if mode != 'experts':
                model, best_ep, XF, XV, _ = train_head(F_RAW, V_RAW, np.concatenate([tr_idx, EXT]), vsets, seed=seed, mode=mode)
                parts = {None: (model, XF, XV)}
            else:
                parts, eps, mu_pos = {}, [], {}
                for g in (0, 1):
                    all_g = np.concatenate([tr_idx[Y_G[tr_idx] == g], EXT[Y_G[EXT] == g]])
                    vs_g = [(Vi[fg[Fi] == g], Fi[fg[Fi] == g], L[fg[Fi] == g]) for Vi, Fi, L in vsets]
                    m_, ep_, XF, XV, _ = train_head(F_RAW, V_RAW, all_g, vs_g, seed=seed, mode='plain'); parts[g] = (m_, XF, XV); eps.append(ep_)
                    Vi, Fi, L = vs_g[0]; mu_pos[g] = same_score(m_, XF[Fi[L == 1]], XV[Vi[L == 1]]).mean()
                best_ep = int(np.mean(eps)); cal = mu_pos[0] - mu_pos[1]            # calibration gap between the two experts' positive scores
            scores = score_sets(parts, vsets, fg)
            e_std, e_gen = [eer(L, s) for (Vi, Fi, L), s in zip(vsets, scores)]
            Vi, Fi, L = vsets[1]; s = scores[1]
            e_pg = np.mean([eer(L[fg[Fi] == g], s[fg[Fi] == g]) for g in (0, 1) if (fg[Fi] == g).sum() > 20 and L[fg[Fi] == g].min() != L[fg[Fi] == g].max()])
            row = pd.DataFrame([dict(combo=name, seed=seed, fold=k, best_ep=best_ep, std=e_std, gender=e_gen, gender_pg=e_pg, cal=cal)])
            row.to_csv(CSV, mode='a', header=not os.path.exists(CSV), index=False); done = pd.concat([done, row])
            print('%-18s seed %d fold %d | ep %3d | std %.2f | gender %.2f | gender per-face-gender %.2f | cal %.3f | %.0f min' % (name, seed, k, best_ep, e_std, e_gen, e_pg, cal, (time.time() - t0) / 60))
R = pd.read_csv(CSV); R['overall'] = (R['std'] + R['gender']) / 2
tab = R.groupby(['combo', 'seed'])[['std', 'gender', 'gender_pg', 'overall', 'cal']].mean().groupby('combo').mean().round(2)
ref = tab.loc['G0_C_ref']
note('WITHIN-GENDER METHODS on system C features (challenge + all MAV, CLIP | R4 + Whisper + NAP-8; 5-fold x 3 seeds)')
note(tab.to_string())
note('NB24 D4 reference for the same recipe was 25.15 (std 19.02 / gender 31.28); G0 here = %.2f (%.2f / %.2f)' % (ref['overall'], ref['std'], ref['gender']))
verdicts = []
for c in tab.index:
    if c == 'G0_C_ref':
        continue
    ok = (tab.loc[c, 'gender'] < ref['gender'] - 0.5) and (tab.loc[c, 'std'] <= ref['std'] + 0.2)
    verdicts.append('%s: gender %+.2f, std %+.2f -> %s' % (c, tab.loc[c, 'gender'] - ref['gender'], tab.loc[c, 'std'] - ref['std'], 'ADOPT (stage 2 = NB20 v5)' if ok else 'reject'))
note('VERDICTS (rule: gender better by >= 0.5 AND std not worse by > 0.2): ' + ' | '.join(verdicts))
print('=' * 70); print('SUMMARY  (copy everything between the lines and paste to Claude)'); print('=' * 70)
for s in SUMMARY:
    print(s)
print('=' * 70)